# 03 消融测试 + 安慰剂测试 (Ablation & Placebo)

## 消融测试
系统性掩码各维度，测量 PC/CI/IDS 变化，识别最大的记忆触发器。

## 安慰剂测试
跨新闻打乱实体+数字，保留日期和结构。通过标准：安慰剂准确率 < 55%。

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed
from src.masking import extract_json_robust
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, load_counterfactual_variants
from src.experiment import run_counterfactual_attack, run_scoring_batch
from src.masking import apply_masking
from src.prompts import scoring_prompt
from src.metrics import prediction_consistency, confidence_invariance, input_dependency_score, composite_leakage_score
from src.display_utils import show_comparison
import json
import numpy as np
import pandas as pd
import random

set_seed()

## 1. 消融测试：逐维度掩码

In [ ]:
test_cases = load_test_cases()
variants = load_counterfactual_variants()
variant_map = {}
for v in variants:
    variant_map.setdefault(v.original_case_id, {})[v.variant_type] = v

client = LLMClient()
output_format = "5-bin"

# Ablation: progressively add masks
ablation_configs = {
    "baseline": MaskingConfig(),
    "+year": MaskingConfig(mask_year=True),
    "+year+entity": MaskingConfig(mask_year=True, mask_entity=True),
    "+year+entity+number": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True),
    "+year+entity+number+sector": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True),
    "full (all+role+cot+constraint)": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True, role_play=True, cot_forced=True, extraction_constraint=True),
}

In [ ]:
def parse_5bin(raw: str) -> dict:
    data = extract_json_robust(raw)
    if not data:
        return {"direction": "neutral", "confidence": 0.5, "distribution": [0.2]*5}
    dist = [data.get(k, 20) for k in ["strong_bear", "weak_bear", "neutral", "weak_bull", "strong_bull"]]
    total = sum(dist)
    dist = [d/total for d in dist] if total > 0 else [0.2]*5
    bull, bear = dist[3]+dist[4], dist[0]+dist[1]
    direction = "up" if bull > bear + 0.1 else ("down" if bear > bull + 0.1 else "neutral")
    return {"direction": direction, "confidence": max(bull, bear, dist[2]), "distribution": dist}

ablation_results = []
for config_name, config in ablation_configs.items():
    print(f"\nRunning: {config_name}")

    orig_responses, cf_responses, task_meta = run_counterfactual_attack(
        client, config, test_cases, variant_map, output_format
    )

    for (tc, vt_name), orig_resp, cf_resp in zip(task_meta, orig_responses, cf_responses):
        orig = parse_5bin(orig_resp.raw_response)
        cf = parse_5bin(cf_resp.raw_response)
        ablation_results.append({
            "config": config_name, "case_id": tc.id, "variant_type": vt_name,
            "orig_dir": orig["direction"], "cf_dir": cf["direction"],
            "orig_conf": orig["confidence"], "cf_conf": cf["confidence"],
            "orig_dist": orig["distribution"], "cf_dist": cf["distribution"],
        })

## 2. 消融结果分析

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

df_abl = pd.DataFrame(ablation_results)
abl_metrics = []
for config_name, group in df_abl.groupby("config"):
    pc = prediction_consistency(group["orig_dir"].tolist(), group["cf_dir"].tolist())
    consistent = [o == c for o, c in zip(group["orig_dir"], group["cf_dir"])]
    ci = confidence_invariance(group["orig_conf"].tolist(), group["cf_conf"].tolist(), consistent)
    ids = input_dependency_score(group["orig_dist"].tolist(), group["cf_dist"].tolist())
    L = composite_leakage_score(pc, ci, ids)
    abl_metrics.append({"config": config_name, "PC": pc, "CI": ci, "IDS": ids, "L": L})

abl_df = pd.DataFrame(abl_metrics)
config_order = list(ablation_configs.keys())
abl_df["config"] = pd.Categorical(abl_df["config"], categories=config_order, ordered=True)
abl_df = abl_df.sort_values("config")

print("Ablation results:")
print(abl_df.to_string(index=False, float_format="%.3f"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(abl_df))
width = 0.2
ax.bar([i - width for i in x], abl_df["PC"], width, label="PC", color="#F44336")
ax.bar(x, abl_df["CI"], width, label="CI", color="#FF9800")
ax.bar([i + width for i in x], abl_df["IDS"], width, label="IDS", color="#4CAF50")
ax.set_xticks(x)
ax.set_xticklabels(abl_df["config"], rotation=45, ha="right")
ax.set_ylabel("Score")
ax.set_title("Ablation: Progressive Masking Impact on Leakage Metrics")
ax.legend()
ax.axhline(y=0.69, color='red', linestyle='--', alpha=0.5, label='Profit Mirage PC baseline')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'ablation_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nMarginal contribution (delta-L per step):")
for i in range(1, len(abl_df)):
    delta = abl_df.iloc[i]["L"] - abl_df.iloc[i-1]["L"]
    print(f"  {abl_df.iloc[i-1]['config']} -> {abl_df.iloc[i]['config']}: dL = {delta:+.3f}")

## 3. 安慰剂测试

跨新闻打乱实体和数字，保留日期和结构。
如果模型真的在分析而非记忆，安慰剂数据的准确率应接近随机（~33% for 3-class）。

通过标准：安慰剂准确率 < 55%

In [ ]:
# Generate placebo: shuffle entities and numbers across news items
all_entities = [e for tc in test_cases for e in tc.key_entities]
all_numbers = [n for tc in test_cases for n in tc.key_numbers]
random.shuffle(all_entities)
random.shuffle(all_numbers)

placebo_texts = []
entity_idx, number_idx = 0, 0

for tc in test_cases:
    text = tc.news.content
    for orig_ent in tc.key_entities:
        if entity_idx < len(all_entities):
            text = text.replace(orig_ent, all_entities[entity_idx])
            entity_idx += 1
    for orig_num in tc.key_numbers:
        if number_idx < len(all_numbers):
            text = text.replace(orig_num, all_numbers[number_idx])
            number_idx += 1
    placebo_texts.append({"case_id": tc.id, "original": tc.news.content, "placebo": text})

for pt in placebo_texts[:3]:
    show_comparison(pt["original"], pt["placebo"], title=f"[{pt['case_id']}] 原文 vs 安慰剂（打乱实体+数字）")

# 并发调用 placebo + real baseline
config = MaskingConfig()
placebo_responses = run_scoring_batch(client, config, [pt["placebo"] for pt in placebo_texts], output_format)
real_responses = run_scoring_batch(client, config, [tc.news.content for tc in test_cases], output_format)

placebo_results = []
for tc, resp in zip(test_cases, placebo_responses):
    parsed = parse_5bin(resp.raw_response)
    placebo_results.append({
        "case_id": tc.id, "expected": tc.expected_direction.value,
        "predicted": parsed["direction"],
        "correct": parsed["direction"] == tc.expected_direction.value,
    })

real_results = []
for tc, resp in zip(test_cases, real_responses):
    parsed = parse_5bin(resp.raw_response)
    real_results.append({
        "case_id": tc.id, "expected": tc.expected_direction.value,
        "predicted": parsed["direction"],
        "correct": parsed["direction"] == tc.expected_direction.value,
    })

## 4. 安慰剂测试结果

In [ ]:
real_acc = sum(r["correct"] for r in real_results) / len(real_results)
placebo_acc = sum(r["correct"] for r in placebo_results) / len(placebo_results)

print(f"Real data accuracy:    {real_acc:.1%}")
print(f"Placebo accuracy:      {placebo_acc:.1%}")
print(f"Difference:            {real_acc - placebo_acc:+.1%}")
print()

if placebo_acc < 0.55:
    print("PASS: Placebo accuracy < 55%")
else:
    print("FAIL: Placebo accuracy >= 55%, model may rely on memorization")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Real", "Placebo"], [real_acc, placebo_acc], color=["#2196F3", "#9E9E9E"])
ax.axhline(y=0.55, color='red', linestyle='--', label='Threshold (55%)')
ax.axhline(y=0.33, color='green', linestyle='--', alpha=0.5, label='Random (33%)')
ax.set_ylabel("Accuracy")
ax.set_title("Placebo Test: Real vs Shuffled Data")
ax.set_ylim(0, 1)
ax.legend()
for bar, val in zip(bars, [real_acc, placebo_acc]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.1%}", ha='center')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'placebo_test.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. 保存结果

In [ ]:
output = {
    "ablation_metrics": abl_metrics,
    "placebo": {"real_accuracy": real_acc, "placebo_accuracy": placebo_acc, "pass": placebo_acc < 0.55},
}
output_path = PROJECT_ROOT / 'data' / 'results' / 'ablation_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)
print(f"Results saved to {output_path}")